# T21 — Composite Presets

Generate multi-domain datasets using built-in presets with pre-configured shared entity mappings.

**What you'll learn:**
- List and inspect available presets
- Generate from a preset in one line
- Verify cross-domain FK integrity
- Compare preset vs ad-hoc composite

## 1. List Available Presets

In [ ]:
from sqllocks_spindle.presets import list_presets, get_preset

for p in list_presets():
    print(f"{p.name:25s} domains={', '.join(p.domains)}")
    print(f"{'':25s} {p.description}\n")

## 2. Generate from Enterprise Preset

In [ ]:
from sqllocks_spindle import Spindle, RetailDomain, HrDomain, FinancialDomain
from sqllocks_spindle.domains.composite import CompositeDomain

preset = get_preset("enterprise")
composite = CompositeDomain(
    domains=[RetailDomain(), HrDomain(), FinancialDomain()],
    shared_entities=preset.shared_entities,
)

result = Spindle().generate(domain=composite, scale="small", seed=42)
print(result.summary())
print(f"\nTables: {sorted(result.table_names)}")

## 3. Verify Cross-Domain FK Integrity

In [ ]:
errors = result.verify_integrity()
print(f"FK integrity errors: {len(errors)}")
for e in errors[:5]:
    print(f"  {e}")

# Check tables by domain
for prefix in ['retail_', 'hr_', 'financial_']:
    tables = [t for t in result.table_names if t.startswith(prefix)]
    total_rows = sum(len(result[t]) for t in tables)
    print(f"\n{prefix[:-1]:12s}: {len(tables)} tables, {total_rows:,} rows")

## 4. Inspect Shared Entities

In [ ]:
print("Enterprise preset shared entities:")
for concept, config in preset.shared_entities.items():
    print(f"\n  {concept}:")
    print(f"    Primary: {config['primary']}")
    if 'links' in config:
        for domain, link in config['links'].items():
            print(f"    {domain}: {link}")

## 5. Try Other Presets

In [ ]:
from sqllocks_spindle.domains.healthcare import HealthcareDomain
from sqllocks_spindle.domains.insurance import InsuranceDomain

hc_preset = get_preset("healthcare_system")
hc_composite = CompositeDomain(
    domains=[HealthcareDomain(), InsuranceDomain(), HrDomain()],
    shared_entities=hc_preset.shared_entities,
)
hc_result = Spindle().generate(domain=hc_composite, scale="small", seed=42)
print(f"Healthcare System: {len(hc_result.table_names)} tables, "
      f"{sum(len(hc_result[t]) for t in hc_result.table_names):,} rows")

## 6. CLI Equivalent

```bash
# List presets
spindle presets

# Generate from preset
spindle composite enterprise --scale small --output ./data/ --format parquet

# Ad-hoc combination
spindle composite retail+hr+financial --scale small --output ./data/
```